In [ ]:
#Cell 1 — auth + connect to BigQuery
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

project_id = "datathon-2026-506819"   # same project the team's tables live in
client = bigquery.Client(project=project_id)
print("Connected to BigQuery")

Connected to BigQuery


In [ ]:
!pip install pycountry -q
import pycountry

In [ ]:
#Cell 2 — upload the CSV from your machine
from google.colab import files
import pandas as pd

uploaded = files.upload()   # picker opens -- select food_prices_for_nutrition.csv
csv_path = list(uploaded.keys())[0]
print("Loaded:", csv_path)

raw = pd.read_csv(csv_path, encoding="latin1")
print("Shape:", raw.shape)
print("\nOriginal columns:")
print(raw.columns.tolist())
raw.head()

Saving food_prices_for_nutrition.csv to food_prices_for_nutrition (2).csv
Loaded: food_prices_for_nutrition (2).csv
Shape: (1733, 60)

Original columns:
['Classification Name', 'Classification Code', 'Country Name', 'Country Code', 'Time', 'Time Code', 'Cost of a healthy diet relative to the cost of sufficient energy from starchy staples [CoHD_CoCA]', 'Cost share for fruits in a least-cost healthy diet [CoHD_f_prop]', 'Cost share for vegetables in a least-cost healthy diet [CoHD_v_prop]', 'Cost share for starchy staples in a least-cost healthy diet [CoHD_ss_prop]', 'Cost share for animal-sourced foods in a least-cost healthy diet [CoHD_asf_prop]', 'Cost share for legumes, nuts and seeds in a least-cost healthy diet [CoHD_lns_prop]', 'Cost share for oils and fats in a least-cost healthy diet [CoHD_of_prop]', 'Cost of fruits relative to the starchy staples in a least-cost healthy diet [CoHD_f_ss]', 'Cost of vegetables relative to the starchy staples in a least-cost healthy diet [CoHD_v_s

,Classification Name,Classification Code,Country Name,Country Code,Time,Time Code,Cost of a healthy diet relative to the cost of sufficient energy from starchy staples [CoHD_CoCA],Cost share for fruits in a least-cost healthy diet [CoHD_f_prop],Cost share for vegetables in a least-cost healthy diet [CoHD_v_prop],Cost share for starchy staples in a least-cost healthy diet [CoHD_ss_prop],...,Population-weighted cost of an energy sufficient diet in PPP dollars [CoCA_PPP_PoP],Population-weighted cost of a nutrient adequate diet in PPP dollars [CoNA_PPP_PoP],Population-weighted cost of a healthy diet in PPP dollars [CoHD_PPP_PoP],Population-weighted cost of fruits in PPP dollars [CoHD_f_PPP_PoP],Population-weighted cost of vegetables in PPP dollars [CoHD_v_PPP_PoP],Population-weighted cost of starchy staples in PPP dollars [CoHD_ss_PPP_PoP],Population-weighted cost of animal source foods in PPP dollars [CoHD_asf_PPP_PoP],"Population-weighted cost of legumes, nuts and seeds in PPP dollars [CoHD_lns_PPP_PoP]",Population-weighted cost of fats and oils in PPP dollars [CoHD_of_PPP_PoP],Population [Pop]
0,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2017.0,YR2017,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2898242
1,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2018.0,YR2018,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2894231
2,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2019.0,YR2019,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2885010
3,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2020.0,YR2020,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2871954
4,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2021.0,YR2021,4.15,0.22,0.17,0.14,...,..,..,..,..,..,..,..,..,..,2849635


In [ ]:
# Cell 3 — sanitize column names for BigQuery
import re

def bq_safe_name(col):
    # prefer the bracketed short code if present, e.g. "... [CoHD_headcount]" -> "CoHD_headcount"
    match = re.search(r"\[([^\[\]]+)\]\s*$", col.strip())
    name = match.group(1) if match else col

    name = re.sub(r"[^0-9a-zA-Z_]+", "_", name)   # spaces, commas, dots, $, parens -> _
    name = re.sub(r"_+", "_", name).strip("_")     # collapse/trim underscores

    if not name:
        name = "col"
    if name[0].isdigit():
        name = "_" + name
    return name

def make_unique(names):
    seen, out = {}, []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}_{seen[n]}")
    return out

clean = raw.copy()
clean.columns = make_unique([bq_safe_name(c) for c in clean.columns])

print("Cleaned columns:")
print(clean.columns.tolist())

assert clean.columns.is_unique, "Duplicate column names after cleaning"
assert all(re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", c) for c in clean.columns), "Invalid BigQuery column name found"

clean.head()

Cleaned columns:
['Classification_Name', 'Classification_Code', 'Country_Name', 'Country_Code', 'Time', 'Time_Code', 'CoHD_CoCA', 'CoHD_f_prop', 'CoHD_v_prop', 'CoHD_ss_prop', 'CoHD_asf_prop', 'CoHD_lns_prop', 'CoHD_of_prop', 'CoHD_f_ss', 'CoHD_v_ss', 'CoHD_asf_ss', 'CoHD_lns_ss', 'CoHD_of_ss', 'CoCA_pov', 'CoNA_pov', 'CoHD_pov', 'CoCA_fexp', 'CoNA_fexp', 'CoHD_fexp', 'CoCA_headcount', 'CoNA_headcount', 'CoHD_headcount', 'CoCA_unafford_n', 'CoNA_unafford_n', 'CoHD_unafford_n', 'CoCA_LCU', 'CoNA_LCU', 'CoHD_LCU', 'CoCA_PPP', 'CoNA_PPP', 'CoHD_PPP', 'CoHD_f_LCU', 'CoHD_v_LCU', 'CoHD_ss_LCU', 'CoHD_asf_LCU', 'CoHD_lns_LCU', 'CoHD_of_LCU', 'NY_GDP_PCAP_PP_CD', 'CoHD_f_PPP', 'NY_GNP_PCAP_PP_CD', 'CoHD_v_PPP', 'CoHD_ss_PPP', 'CoHD_asf_PPP', 'CoHD_lns_PPP', 'CoHD_of_PPP', 'CoCA_PPP_PoP', 'CoNA_PPP_PoP', 'CoHD_PPP_PoP', 'CoHD_f_PPP_PoP', 'CoHD_v_PPP_PoP', 'CoHD_ss_PPP_PoP', 'CoHD_asf_PPP_PoP', 'CoHD_lns_PPP_PoP', 'CoHD_of_PPP_PoP', 'Pop']


,Classification_Name,Classification_Code,Country_Name,Country_Code,Time,Time_Code,CoHD_CoCA,CoHD_f_prop,CoHD_v_prop,CoHD_ss_prop,...,CoCA_PPP_PoP,CoNA_PPP_PoP,CoHD_PPP_PoP,CoHD_f_PPP_PoP,CoHD_v_PPP_PoP,CoHD_ss_PPP_PoP,CoHD_asf_PPP_PoP,CoHD_lns_PPP_PoP,CoHD_of_PPP_PoP,Pop
0,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2017.0,YR2017,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2898242
1,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2018.0,YR2018,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2894231
2,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2019.0,YR2019,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2885010
3,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2020.0,YR2020,..,..,..,..,...,..,..,..,..,..,..,..,..,..,2871954
4,Food Prices for Nutrition 5.0,FPN 5.0,Albania,ALB,2021.0,YR2021,4.15,0.22,0.17,0.14,...,..,..,..,..,..,..,..,..,..,2849635


In [ ]:
# Cell 4 — check for stray rows/footnotes before upload
# World Bank DataBank exports often end with footer rows (source notes, blank lines)
# that break the schema. Check the tail before loading.
print(clean.tail(10))
print("\nRows with a null Country_Code (likely footer junk):")
display(clean[clean["Country_Code"].isna()])





                                Classification_Name Classification_Code  \
1723                  Food Prices for Nutrition 5.0             FPN 5.0   
1724                  Food Prices for Nutrition 5.0             FPN 5.0   
1725                  Food Prices for Nutrition 5.0             FPN 5.0   
1726                  Food Prices for Nutrition 5.0             FPN 5.0   
1727                  Food Prices for Nutrition 5.0             FPN 5.0   
1728                                            NaN                 NaN   
1729                                            NaN                 NaN   
1730                                            NaN                 NaN   
1731  Data from database: Food Prices for Nutrition                 NaN   
1732                       Last Updated: 07/21/2026                 NaN   

     Country_Name Country_Code    Time Time_Code CoHD_CoCA CoHD_f_prop  \
1723     Zimbabwe          ZWE  2021.0    YR2021      2.89        0.12   
1724     Zimbabwe         

,Classification_Name,Classification_Code,Country_Name,Country_Code,Time,Time_Code,CoHD_CoCA,CoHD_f_prop,CoHD_v_prop,CoHD_ss_prop,...,CoCA_PPP_PoP,CoNA_PPP_PoP,CoHD_PPP_PoP,CoHD_f_PPP_PoP,CoHD_v_PPP_PoP,CoHD_ss_PPP_PoP,CoHD_asf_PPP_PoP,CoHD_lns_PPP_PoP,CoHD_of_PPP_PoP,Pop
1728,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1730,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1731,Data from database: Food Prices for Nutrition,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1732,Last Updated: 07/21/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#If that last check shows footer rows, drop them:
clean = clean[clean["Country_Code"].notna()].reset_index(drop=True)

In [ ]:
# Cell 5 — load into BigQuery
dataset_name = "Food_Prices_for_Nutrition"
table_name = "Food_Prices_for_Nutrition"

dataset_ref = f"{project_id}.{dataset_name}"
table_ref = f"{dataset_ref}.{table_name}"

client.create_dataset(dataset_ref, exists_ok=True)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True,
)

load_job = client.load_table_from_dataframe(clean, table_ref, job_config=job_config)
load_job.result()

table = client.get_table(table_ref)
print(f"Loaded {table.num_rows} rows into {table_ref}")
for field in table.schema:
    print(f"  {field.name} --> {field.field_type}")

Loaded 1728 rows into datathon-2026-506819.Food_Prices_for_Nutrition.Food_Prices_for_Nutrition
  Classification_Name --> STRING
  Classification_Code --> STRING
  Country_Name --> STRING
  Country_Code --> STRING
  Time --> FLOAT
  Time_Code --> STRING
  CoHD_CoCA --> STRING
  CoHD_f_prop --> STRING
  CoHD_v_prop --> STRING
  CoHD_ss_prop --> STRING
  CoHD_asf_prop --> STRING
  CoHD_lns_prop --> STRING
  CoHD_of_prop --> STRING
  CoHD_f_ss --> STRING
  CoHD_v_ss --> STRING
  CoHD_asf_ss --> STRING
  CoHD_lns_ss --> STRING
  CoHD_of_ss --> STRING
  CoCA_pov --> STRING
  CoNA_pov --> STRING
  CoHD_pov --> STRING
  CoCA_fexp --> STRING
  CoNA_fexp --> STRING
  CoHD_fexp --> STRING
  CoCA_headcount --> STRING
  CoNA_headcount --> STRING
  CoHD_headcount --> STRING
  CoCA_unafford_n --> STRING
  CoNA_unafford_n --> STRING
  CoHD_unafford_n --> STRING
  CoCA_LCU --> STRING
  CoNA_LCU --> STRING
  CoHD_LCU --> STRING
  CoCA_PPP --> STRING
  CoNA_PPP --> STRING
  CoHD_PPP --> STRING
  CoHD_f_L

In [ ]:
table = client.get_table(table_ref)
for field in table.schema:
    print(field.name)

Classification_Name
Classification_Code
Country_Name
Country_Code
Time
Time_Code
CoHD_CoCA
CoHD_f_prop
CoHD_v_prop
CoHD_ss_prop
CoHD_asf_prop
CoHD_lns_prop
CoHD_of_prop
CoHD_f_ss
CoHD_v_ss
CoHD_asf_ss
CoHD_lns_ss
CoHD_of_ss
CoCA_pov
CoNA_pov
CoHD_pov
CoCA_fexp
CoNA_fexp
CoHD_fexp
CoCA_headcount
CoNA_headcount
CoHD_headcount
CoCA_unafford_n
CoNA_unafford_n
CoHD_unafford_n
CoCA_LCU
CoNA_LCU
CoHD_LCU
CoCA_PPP
CoNA_PPP
CoHD_PPP
CoHD_f_LCU
CoHD_v_LCU
CoHD_ss_LCU
CoHD_asf_LCU
CoHD_lns_LCU
CoHD_of_LCU
NY_GDP_PCAP_PP_CD
CoHD_f_PPP
NY_GNP_PCAP_PP_CD
CoHD_v_PPP
CoHD_ss_PPP
CoHD_asf_PPP
CoHD_lns_PPP
CoHD_of_PPP
CoCA_PPP_PoP
CoNA_PPP_PoP
CoHD_PPP_PoP
CoHD_f_PPP_PoP
CoHD_v_PPP_PoP
CoHD_ss_PPP_PoP
CoHD_asf_PPP_PoP
CoHD_lns_PPP_PoP
CoHD_of_PPP_PoP
Pop


In [ ]:
# Dataset uses "time" instead of "year"

client.query(f"""
ALTER TABLE `{table_ref}`
RENAME COLUMN Time TO Year
""").result()

client.query(f"""
ALTER TABLE `{table_ref}`
RENAME COLUMN Time_Code TO Year_Code
""").result()

print("Renamed.")

Renamed.


In [ ]:
# Cell 6 — verify by querying it back
check = client.query(f"""
SELECT Country_Name, Country_Code, Year, CoHD_headcount, NY_GDP_PCAP_PP_CD, Pop
FROM `{table_ref}`
ORDER BY Country_Name, Year
LIMIT 10
""").to_dataframe()

display(check)

,Country_Name,Country_Code,Year,CoHD_headcount,NY_GDP_PCAP_PP_CD,Pop
0,Albania,ALB,2017.0,29.2,..,2898242
1,Albania,ALB,2018.0,20.9,..,2894231
2,Albania,ALB,2019.0,18.4,..,2885010
3,Albania,ALB,2020.0,17.6,..,2871954
4,Albania,ALB,2021.0,16,..,2849635
5,Albania,ALB,2022.0,14.2,..,2827608
6,Albania,ALB,2023.0,13.9,..,2811655
7,Albania,ALB,2024.0,12.6,..,2791765
8,Albania,ALB,2025.0,11.9,..,2771508
9,Algeria,DZA,2017.0,41.3,..,41689299


In [ ]:
gdp_check = client.query(f"""
SELECT
  COUNTIF(NY_GDP_PCAP_PP_CD != '..') AS rows_with_gdp,
  COUNTIF(NY_GDP_PCAP_PP_CD = '..') AS rows_missing_gdp,
  COUNT(*) AS total_rows
FROM `{table_ref}`
""").to_dataframe()

display(gdp_check)

,rows_with_gdp,rows_missing_gdp,total_rows
0,0,1728,1728


In [ ]:
gni_check = client.query(f"""
SELECT COUNTIF(NY_GNP_PCAP_PP_CD != '..') AS rows_with_gni, COUNT(*) AS total_rows
FROM `{table_ref}`
""").to_dataframe()
display(gni_check)

,rows_with_gni,total_rows
0,0,1728


In [ ]:
uploaded_gdp = files.upload()   # select GDP_Per_Capita_World_Bank.csv
gdp_csv_path = list(uploaded_gdp.keys())[0]
print("Loaded:", gdp_csv_path)


Saving GDP_Per_Capita_World_Bank.csv to GDP_Per_Capita_World_Bank (1).csv
Loaded: GDP_Per_Capita_World_Bank (1).csv


In [ ]:
gdp_raw = pd.read_csv(gdp_csv_path, skiprows=4, encoding="latin1")
print("Shape:", gdp_raw.shape)
print(gdp_raw.columns.tolist()[:10])
gdp_raw.head()

Shape: (265, 71)
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965']


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,38213.289239,39378.228411,39754.560072,30408.827090,35207.170909,39083.461640,42783.248907,45428.002728,NaN,NaN
1,Africa Eastern and Southern,AFE,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,4223.188050,4229.839634,4200.531044,3978.133502,4049.969513,4098.126192,4076.224898,4099.889812,4164.679177,NaN
2,Afghanistan,AFG,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,2952.998916,2902.392113,2927.245144,2769.685745,2144.166570,1981.710168,1983.812620,1964.452289,NaN,NaN
3,Africa Western and Central,AFW,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,6009.516114,6026.527572,6070.284769,5698.766702,5702.134886,5818.483224,5886.796636,6009.084662,6139.587294,NaN
4,Angola,AGO,"GDP per capita, PPP (constant 2021 internation...",NY.GDP.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,10565.167648,10154.838923,9712.979449,8904.949006,8720.187687,8750.993630,8597.538788,8752.797471,8760.190820,NaN


In [ ]:
gdp_long = gdp_raw.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year", value_name="GDP_Per_Capita_PPP"
)

gdp_long["Year"] = pd.to_numeric(gdp_long["Year"], errors="coerce")
gdp_clean = gdp_long.dropna(subset=["Year", "GDP_Per_Capita_PPP"]).copy()
gdp_clean["Year"] = gdp_clean["Year"].astype(int)

gdp_clean = gdp_clean.rename(columns={
    "Country Name": "Country_Name", "Country Code": "Country_Code"
})[["Country_Name", "Country_Code", "Year", "GDP_Per_Capita_PPP"]]

print("Shape after cleaning:", gdp_clean.shape)
gdp_clean.head()

Shape after cleaning: (8669, 4)


,Country_Name,Country_Code,Year,GDP_Per_Capita_PPP
7950,Aruba,ABW,1990,33146.670548
7951,Africa Eastern and Southern,AFE,1990,3520.019233
7953,Africa Western and Central,AFW,1990,4165.726363
7954,Angola,AGO,1990,8366.412716
7955,Albania,ALB,1990,5560.833935


In [ ]:
gdp_table_name = "GDP_Per_Capita"
gdp_table_ref = f"{project_id}.{dataset_name}.{gdp_table_name}"

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE", autodetect=True)
load_job = client.load_table_from_dataframe(gdp_clean, gdp_table_ref, job_config=job_config)
load_job.result()

table = client.get_table(gdp_table_ref)
print(f"Loaded {table.num_rows} rows into {gdp_table_ref}")

Loaded 8669 rows into datathon-2026-506819.Food_Prices_for_Nutrition.GDP_Per_Capita


In [ ]:
check_gdp = client.query(f"""
SELECT Country_Name, Country_Code, Year, GDP_Per_Capita_PPP
FROM `{gdp_table_ref}`
ORDER BY Country_Name, Year
LIMIT 10
""").to_dataframe()
display(check_gdp)

,Country_Name,Country_Code,Year,GDP_Per_Capita_PPP
0,Afghanistan,AFG,2000,1617.826475
1,Afghanistan,AFG,2001,1454.110782
2,Afghanistan,AFG,2002,1774.308743
3,Afghanistan,AFG,2003,1815.928200
4,Afghanistan,AFG,2004,1776.918207
5,Afghanistan,AFG,2005,1908.114782
6,Afghanistan,AFG,2006,1929.723897
7,Afghanistan,AFG,2007,2155.353068
8,Afghanistan,AFG,2008,2191.504356
9,Afghanistan,AFG,2009,2565.022086


In [ ]:
gdp_master = client.query(f"""
SELECT Country_Name, Country_Code, Year, GDP_Per_Capita_PPP
FROM `{project_id}.Food_Prices_for_Nutrition.GDP_Per_Capita`
""").to_dataframe()

gdp_master = gdp_master.rename(columns={
    "Country_Name": "country", "Country_Code": "country_code",
    "Year": "year", "GDP_Per_Capita_PPP": "gdp_per_capita_ppp"
})

valid_iso3 = {c.alpha_3 for c in pycountry.countries} | {"XKX"}
dropped = gdp_master.loc[~gdp_master["country_code"].isin(valid_iso3), "country"].unique()
print("Dropped as non-national:", sorted(dropped))

gdp_master = gdp_master[gdp_master["country_code"].isin(valid_iso3)].reset_index(drop=True)
gdp_master["year"] = gdp_master["year"].astype(int)

print("duplicate country-year rows:", gdp_master.duplicated(subset=["country_code","year"]).sum())
gdp_master.head()

Dropped as non-national: ['Africa Eastern and Southern', 'Africa Western and Central', 'Arab World', 'Caribbean small states', 'Central Europe and the Baltics', 'Early-demographic dividend', 'East Asia & Pacific', 'East Asia & Pacific (IDA & IBRD countries)', 'East Asia & Pacific (excluding high income)', 'Euro area', 'Europe & Central Asia', 'Europe & Central Asia (IDA & IBRD countries)', 'Europe & Central Asia (excluding high income)', 'European Union', 'Heavily indebted poor countries (HIPC)', 'High income', 'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only', 'IDA total', 'Late-demographic dividend', 'Latin America & Caribbean', 'Latin America & Caribbean (excluding high income)', 'Latin America & the Caribbean (IDA & IBRD countries)', 'Least developed countries: UN classification', 'Low & middle income', 'Low income', 'Lower middle income', 'Middle East, North Africa, Afghanistan & Pakistan', 'Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)', 'Middle East, Nort

,country,country_code,year,gdp_per_capita_ppp
0,Aruba,ABW,1990,33146.670548
1,Aruba,ABW,1991,34079.230463
2,Aruba,ABW,1992,34458.142298
3,Aruba,ABW,1993,34627.743475
4,Aruba,ABW,1994,35580.533978


In [ ]:
print("Kosovo" in gdp_raw["Country Name"].values)

True


In [ ]:
print(gdp_raw.loc[gdp_raw["Country Name"] == "Kosovo", "Country Code"])

260    XKX
Name: Country Code, dtype: object


In [ ]:
kosovo_stunting = client.query(f"""
SELECT DISTINCT Country_Name, Country_Code
FROM `{project_id}.Prevalence_of_stunting.Prevalence_of_stunting_clean`
WHERE Country_Name LIKE '%Kosovo%'
""").to_dataframe()
print("Stunting table:", kosovo_stunting.to_dict('records'))

kosovo_women = client.query(f"""
SELECT DISTINCT Location
FROM `{project_id}.Prevalence_of_anaemia_in_women_of_reproductive_age.Prevalence_of_anaemia_in_women_of_reproductive_age_Aged_15_49`
WHERE Location LIKE '%Kosovo%'
""").to_dataframe()
print("Women anaemia table:", kosovo_women.to_dict('records'))

Stunting table: [{'Country_Name': 'Kosovo', 'Country_Code': 'XKX'}]
Women anaemia table: []


In [ ]:
fpn_master = client.query(f"""
SELECT Country_Name, Country_Code, Year, CoHD_headcount, Pop
FROM `{project_id}.Food_Prices_for_Nutrition.Food_Prices_for_Nutrition`
""").to_dataframe()

fpn_master = fpn_master.rename(columns={
    "Country_Name": "country", "Country_Code": "country_code",
    "Year": "year", "CoHD_headcount": "pct_cannot_afford_diet", "Pop": "population"
})

valid_iso3 = {c.alpha_3 for c in pycountry.countries} | {"XKX"}
dropped = fpn_master.loc[~fpn_master["country_code"].isin(valid_iso3), "country"].unique()
print("Dropped as non-national:", sorted(dropped))

fpn_master = fpn_master[fpn_master["country_code"].isin(valid_iso3)].reset_index(drop=True)
fpn_master = fpn_master.dropna(subset=["year"]).copy()
fpn_master["year"] = fpn_master["year"].astype(int)

print("duplicate country-year rows:", fpn_master.duplicated(subset=["country_code","year"]).sum())
fpn_master.head()

Dropped as non-national: ['Bonaire', 'East Asia & Pacific', 'Europe & Central Asia', 'High-income', 'Latin America & Caribbean', 'Low-income', 'Lower-middle-income', 'Middle East & North Africa', 'North America', 'South Asia', 'Sub-Saharan Africa', 'Upper-middle-income', 'WORLD (Benchmark)']
duplicate country-year rows: 0


,country,country_code,year,pct_cannot_afford_diet,population
0,Albania,ALB,2017,29.2,2898242
1,Albania,ALB,2018,20.9,2894231
2,Albania,ALB,2019,18.4,2885010
3,Albania,ALB,2020,17.6,2871954
4,Albania,ALB,2021,16,2849635


In [ ]:
print(fpn_master.loc[fpn_master["country"] == "Bonaire", "country_code"].tolist())

[]


In [ ]:
bonaire_check = client.query(f"""
SELECT DISTINCT Country_Name, Country_Code
FROM `{project_id}.Prevalence_of_stunting.Prevalence_of_stunting_clean`
WHERE Country_Name LIKE '%Bonaire%'
""").to_dataframe()
print(bonaire_check.to_dict('records'))

[]


In [ ]:
client.query("""
ALTER TABLE `datathon-2026-506819.GDP_Per_Capita.GDP_Per_Capita_copy`
RENAME TO GDP_Per_Capita
""").result()
print("Renamed.")

Renamed.
